# Taxonomic Relevance Evaluation

This notebook evaluates a problem in the current species scoring setup: the manual annotations often encode dataset relevance through coarse taxonomic group and richness information, while the extraction output often enumerates detailed taxa. That creates large numbers of false positives and false negatives even when the extraction is arguably useful for screening dataset relevance.

A few concrete patterns illustrate the mismatch:

- **Count signal in manual annotation, enumerated taxa in prediction:** ground truth may say `73 weevil species`, while the model returns a long list of 73 named weevils. Raw `species` comparison treats that as a bad miss even though the prediction captures most of the screening signal.
- **Coarse group in manual annotation, scientific names in prediction:** ground truth may say `Caribou`, while the model returns `Rangifer tarandus` plus a subspecies or population label. This is partly a naming mismatch, not necessarily a relevance failure.
- **Broad taxonomic basket in manual annotation, mixed detailed taxa in prediction:** ground truth may record `12 mammal, 199 ground-dwelling beetles, 240 flying beetles`, while the model predicts many specific mammals and beetles. The prediction may be directionally useful, but not in the same representation as the annotation.
- **Community label in manual annotation, organism list in prediction:** ground truth may say `benthic intertidal community`, while the model outputs annelids, molluscs, and algae. That captures biological content, but only some derived views can connect it back to the broader dataset-level concept.

These examples show that the annotation and extraction pipelines often capture different taxonomic signals: coarse relevance labels, richness summaries, vernacular names, scientific names, and explicit taxon lists.

## Questions

1. Which mismatch patterns are evaluation artifacts rather than extraction failures?
2. Does a count-focused derived field recover signal from cases like `73 weevil species` vs 73 enumerated taxa?
3. Does GBIF enrichment improve comparison for vernacular/scientific mismatches?
4. Which derived fields reflect dataset relevance better than the baseline `species` metric?

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from llm_metadata.schemas.data_paper import RunArtifact
from llm_metadata.schemas.fuster_features import DatasetFeaturesExtraction
from llm_metadata.taxonomy_eval import (
    DEFAULT_TAXONOMY_FIELDS,
    build_ground_truth_by_id,
    enrich_indexed_models,
    evaluate_taxonomy_fields,
)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

# Update RUN_GLOB to point to the run artifact for the next experiment.
RUN_GLOB = "*.json"
GT_PATH = PROJECT_ROOT / "data/dataset_092624_validated.xlsx"

candidates = sorted((PROJECT_ROOT / "artifacts/runs").glob(RUN_GLOB))
RUN_ARTIFACT_PATH = candidates[0] if candidates else None

if not RUN_ARTIFACT_PATH or not RUN_ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f"Could not find run artifact. Update RUN_GLOB. Tried: {RUN_ARTIFACT_PATH}"
    )

artifact = RunArtifact.load_json(RUN_ARTIFACT_PATH)
allowed_ids = [record.gt_record_id for record in artifact.records]
pred_by_id = artifact.predictions_by_id(DatasetFeaturesExtraction)
true_by_id = build_ground_truth_by_id(GT_PATH, allowed_ids=allowed_ids)

display(Markdown(
    f"**Run artifact:** `{RUN_ARTIFACT_PATH}`  \n"
    f"**Records loaded:** GT={len(true_by_id)} | Pred={len(pred_by_id)}"
))

In [ ]:
FIELDS = DEFAULT_TAXONOMY_FIELDS

report = evaluate_taxonomy_fields(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=FIELDS,
    include_gbif=True,
    gbif_confidence_threshold=80,
)

metrics_df = report.metrics_to_pandas().sort_values(
    ["f1", "precision", "recall"], ascending=False
)
display(Markdown("### Strategy metrics"))
display(metrics_df)

In [ ]:
detail_df = report.to_pandas()

for field in FIELDS:
    mismatches = detail_df[(detail_df["field"] == field) & (~detail_df["match"])]
    if mismatches.empty:
        continue
    display(Markdown(f"### {field} mismatches"))
    display(
        mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(30)
    )

## Analysis (from archived experiment)

For this run, the derived taxonomic views are useful, but they answer different questions and should not be collapsed into a single replacement metric.

- `taxon_broad_group_labels` remains the strongest relevance-oriented view among strategies that cover all records. If one taxonomic score had to stand in for dataset-level relevance, this is the most defensible candidate.
- `species_richness_counts` recovers a different signal: the annotation sometimes encodes richness as a count-bearing group mention, while the model responds with an explicit list of taxa. The count projection helps with those cases.
- `species_richness_group_keys` remains too strict for current model behavior.
- `gbif_keys` mostly helps when the disagreement is naming style rather than relevance.

## Discussion

The current results support keeping raw `species` evaluation, but not treating it as the only taxonomic score.

A practical reading: use `taxon_broad_group_labels` when the question is whole-dataset screening relevance, use `species` and `gbif_keys` when the question is literal taxon recovery or naming normalization, and use the richness strategies as narrower diagnostics.

Concrete next steps for prompt and evaluation work:

- Clean the extraction prompt to reduce non-taxon goop in species output.
- If the adjudication supports it, either add a standard broad-group comparison view to `prompt_eval` reports or prompt the model to emit structured taxon objects directly.

Open questions to revisit after more runs:

- Which baseline `species` mismatches become reasonable matches under derived views across abstract vs full-text modes?
- Does `taxon_broad_group_labels` stay useful after auditing false positives from GBIF-to-group mapping?
- Should the metrics table surface an explicit `applicable_n` or coverage percentage?

## Personal observations (from archived experiment)

There are many different signals captured in the groundtruth species values:

* **Species richness:** Number of organisms associated to a taxonomic group or habitat community. As single value or list.
* **Species/taxonomic identity:** Vernacular or scientific name of an organism, sometimes both. As single value or list.
* **Group:** Taxonomic group or habitat specific community. From species annotation, can be either isolated or joint to richness. As single value or list.

Right now: Both true and preds are mixed bags. Current investigation helped uncover eval strategy and relevant enrichment for each.

**Road to success:**

Taxonomic signal decoupled using distinct evaluated fields, either from LLM extraction and/or species processing. Single evaluation for each.

* **Contracts:** New Extraction and Evaluation models including evaluated taxonomy fields + construction methods from Features models. Difference between extraction and evaluation is added gbif_keys from species.
* **Extraction:** Improved prompt with signal-specific instructions and positive/negative examples from previous extractions. Wired prompt_eval. Improved species prompt to remove individual mentions and clearer instructions from signal observation. Strip non-taxon goop. Taxonomic signals should not be considered mutually exclusive.
* **Evals:** Per-taxonomic-signal eval strategy.
    - Species field kept with enhanced_species strat.
    - gbif_keys: Useful. Applicability is key.
    - 3 richness signals: `species_richness_counts`, `species_richness_group_keys`, `taxon_broad_group_labels`.